# PSOD: Real-World Case Studies

## Practical Applications of Outlier Detection

This notebook demonstrates PSOD on real-world scenarios:

1. **Credit Card Fraud Detection**
2. **Network Intrusion Detection**
3. **Manufacturing Defect Detection**
4. **Healthcare Anomaly Detection**
5. **IoT Sensor Monitoring**

Each case study includes:
- Problem description
- Data preprocessing
- Feature engineering
- Model training and evaluation
- Results interpretation
- Deployment considerations

## Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)

# For development
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent.parent / 'src'))

from psod import (
    PSOD,
    evaluate_outlier_detection,
    save_model,
    load_model,
    compute_feature_importance
)
from psod.visualization import (
    plot_outlier_scores,
    plot_outliers_scatter,
    plot_feature_contributions,
    create_outlier_dashboard
)

print("Setup complete!")

## Case Study 1: Credit Card Fraud Detection

### Problem Description

Credit card fraud is a major concern for financial institutions. We need to detect fraudulent transactions in real-time while minimizing false positives that could inconvenience legitimate customers.

### Challenge:
- Highly imbalanced dataset (fraud is rare)
- Real-time detection required
- High cost of false negatives
- Need for interpretability

In [ ]:
# Generate synthetic credit card transaction data
np.random.seed(42)
n_transactions = 10000
n_fraud = 100  # 1% fraud rate

# Normal transactions
normal_transactions = {
    'amount': np.random.lognormal(mean=4, sigma=1, size=n_transactions - n_fraud),
    'hour': np.random.choice(range(6, 23), size=n_transactions - n_fraud),  # Typical hours
    'distance_from_home': np.random.exponential(scale=20, size=n_transactions - n_fraud),
    'distance_from_last': np.random.exponential(scale=10, size=n_transactions - n_fraud),
    'avg_amount_last_10': np.random.lognormal(mean=4, sigma=0.8, size=n_transactions - n_fraud),
    'num_transactions_today': np.random.poisson(lam=3, size=n_transactions - n_fraud),
}

# Fraudulent transactions (different patterns)
fraud_transactions = {
    'amount': np.random.uniform(500, 5000, size=n_fraud),  # Larger amounts
    'hour': np.random.choice(range(0, 6), size=n_fraud),  # Unusual hours
    'distance_from_home': np.random.uniform(100, 1000, size=n_fraud),  # Far from home
    'distance_from_last': np.random.uniform(50, 500, size=n_fraud),  # Large jumps
    'avg_amount_last_10': np.random.lognormal(mean=3.5, sigma=0.5, size=n_fraud),
    'num_transactions_today': np.random.poisson(lam=1, size=n_fraud),
}

# Combine data
df_fraud = pd.DataFrame({
    'amount': np.concatenate([normal_transactions['amount'], fraud_transactions['amount']]),
    'hour': np.concatenate([normal_transactions['hour'], fraud_transactions['hour']]),
    'distance_from_home': np.concatenate([normal_transactions['distance_from_home'], 
                                           fraud_transactions['distance_from_home']]),
    'distance_from_last': np.concatenate([normal_transactions['distance_from_last'], 
                                          fraud_transactions['distance_from_last']]),
    'avg_amount_last_10': np.concatenate([normal_transactions['avg_amount_last_10'], 
                                          fraud_transactions['avg_amount_last_10']]),
    'num_transactions_today': np.concatenate([normal_transactions['num_transactions_today'], 
                                              fraud_transactions['num_transactions_today']]),
})

# Add categorical features
df_fraud['merchant_category'] = np.random.choice(
    ['retail', 'grocery', 'gas', 'restaurant', 'online'], 
    size=n_transactions
)
df_fraud['card_type'] = np.random.choice(['credit', 'debit'], size=n_transactions)

# Ground truth
y_true_fraud = np.array([0] * (n_transactions - n_fraud) + [1] * n_fraud)

print(f"Dataset shape: {df_fraud.shape}")
print(f"Fraud rate: {100 * n_fraud / n_transactions:.2f}%")
print(f"\nSample transactions:\n{df_fraud.head()}")
print(f"\nFeature statistics:\n{df_fraud.describe()}")

In [ ]:
# Feature engineering
df_fraud_features = df_fraud.copy()

# Create derived features
df_fraud_features['amount_vs_avg'] = df_fraud_features['amount'] / (df_fraud_features['avg_amount_last_10'] + 1)
df_fraud_features['is_night'] = (df_fraud_features['hour'] < 6).astype(int)
df_fraud_features['is_high_amount'] = (df_fraud_features['amount'] > df_fraud_features['amount'].quantile(0.95)).astype(int)
df_fraud_features['distance_ratio'] = df_fraud_features['distance_from_last'] / (df_fraud_features['distance_from_home'] + 1)

# Cyclic encoding for hour
df_fraud_features['hour_sin'] = np.sin(2 * np.pi * df_fraud_features['hour'] / 24)
df_fraud_features['hour_cos'] = np.cos(2 * np.pi * df_fraud_features['hour'] / 24)

print(f"Engineered features: {df_fraud_features.shape[1]}")
print(f"New features: {[col for col in df_fraud_features.columns if col not in df_fraud.columns]}")

In [ ]:
# Train PSOD detector
detector_fraud = PSOD(
    cat_columns=['merchant_category', 'card_type'],
    min_cols_chosen=0.5,
    max_cols_chosen=1.0,
    stdevs_to_outlier=2.5,  # More conservative for fraud
    transform_algorithm='yeo-johnson',
    contamination=0.01,  # Expected 1% fraud
    random_seed=42
)

print("Training fraud detection model...")
scores_fraud = detector_fraud.fit_predict(df_fraud_features, return_class=False)
labels_fraud = detector_fraud.fit_predict(df_fraud_features, return_class=True)

print("\nDetection complete!")
print(f"Flagged transactions: {sum(labels_fraud)}")
print(f"Actual fraud: {sum(y_true_fraud)}")

In [ ]:
# Evaluate performance
metrics_fraud = evaluate_outlier_detection(y_true_fraud, labels_fraud, scores_fraud)

print("\n" + "=" * 50)
print("FRAUD DETECTION PERFORMANCE")
print("=" * 50)
print(f"Precision:  {metrics_fraud['precision']:.3f}  (of flagged, how many are fraud?)")
print(f"Recall:     {metrics_fraud['recall']:.3f}  (of fraud, how many detected?)")
print(f"F1-Score:   {metrics_fraud['f1']:.3f}")
print(f"ROC-AUC:    {metrics_fraud['roc_auc']:.3f}")
print(f"PR-AUC:     {metrics_fraud['pr_auc']:.3f}")
print("=" * 50)

# Calculate business metrics
true_positives = sum((labels_fraud == 1) & (y_true_fraud == 1))
false_positives = sum((labels_fraud == 1) & (y_true_fraud == 0))
false_negatives = sum((labels_fraud == 0) & (y_true_fraud == 1))

avg_fraud_amount = df_fraud.loc[y_true_fraud == 1, 'amount'].mean()
fraud_prevented = true_positives * avg_fraud_amount
fraud_missed = false_negatives * avg_fraud_amount

print(f"\nBusiness Impact:")
print(f"  Fraud prevented: ${fraud_prevented:,.2f}")
print(f"  Fraud missed: ${fraud_missed:,.2f}")
print(f"  Customer inconveniences (FP): {false_positives}")

In [ ]:
# Visualize results
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Score distribution
plot_outlier_scores(scores_fraud, labels_fraud, ax=axes[0, 0], 
                   title='Fraud Detection Scores')

# Amount vs distance scatter
normal_mask = labels_fraud == 0
fraud_mask = labels_fraud == 1

axes[0, 1].scatter(df_fraud.loc[normal_mask, 'amount'], 
                   df_fraud.loc[normal_mask, 'distance_from_home'],
                   c='blue', alpha=0.5, s=20, label='Normal')
axes[0, 1].scatter(df_fraud.loc[fraud_mask, 'amount'], 
                   df_fraud.loc[fraud_mask, 'distance_from_home'],
                   c='red', alpha=0.8, s=50, marker='X', label='Fraud')
axes[0, 1].set_xlabel('Transaction Amount ($)')
axes[0, 1].set_ylabel('Distance from Home (km)')
axes[0, 1].set_title('Amount vs Distance from Home')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# Feature importance
feature_imp = compute_feature_importance(detector_fraud, df_fraud_features)
top_features = dict(sorted(feature_imp.items(), key=lambda x: x[1], reverse=True)[:10])

axes[1, 0].barh(list(top_features.keys()), list(top_features.values()), color='steelblue')
axes[1, 0].set_xlabel('Importance')
axes[1, 0].set_title('Top 10 Features for Fraud Detection')
axes[1, 0].grid(True, alpha=0.3, axis='x')

# Transaction hour distribution
axes[1, 1].hist([df_fraud.loc[y_true_fraud == 0, 'hour'], 
                 df_fraud.loc[y_true_fraud == 1, 'hour']], 
                bins=24, label=['Normal', 'Fraud'], alpha=0.7)
axes[1, 1].set_xlabel('Hour of Day')
axes[1, 1].set_ylabel('Count')
axes[1, 1].set_title('Transaction Distribution by Hour')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

In [ ]:
# Save model for production deployment
save_model(detector_fraud, 'fraud_detector.pkl')
print("Fraud detection model saved for deployment!")

# Example: Score a new transaction
new_transaction = pd.DataFrame([{
    'amount': 2500,
    'hour': 3,
    'distance_from_home': 500,
    'distance_from_last': 300,
    'avg_amount_last_10': 85,
    'num_transactions_today': 1,
    'merchant_category': 'online',
    'card_type': 'credit',
    'amount_vs_avg': 2500 / 85,
    'is_night': 1,
    'is_high_amount': 1,
    'distance_ratio': 300 / 500,
    'hour_sin': np.sin(2 * np.pi * 3 / 24),
    'hour_cos': np.cos(2 * np.pi * 3 / 24)
}])

loaded_detector = load_model('fraud_detector.pkl')
fraud_score = loaded_detector.predict(new_transaction, return_class=False)[0]
is_fraud = loaded_detector.predict(new_transaction, return_class=True)[0]

print(f"\nNew transaction:")
print(f"  Fraud score: {fraud_score:.4f}")
print(f"  Flagged as fraud: {bool(is_fraud)}")
print(f"  Recommendation: {'BLOCK' if is_fraud else 'APPROVE'}")

## Case Study 2: Network Intrusion Detection

### Problem Description

Detect malicious network traffic and potential intrusions in a computer network.

### Challenge:
- High-dimensional network features
- Various attack types
- Real-time detection required
- Minimal false alarms

In [ ]:
# Generate synthetic network traffic data
np.random.seed(42)
n_connections = 5000
n_attacks = 150  # 3% attack rate

# Normal traffic
normal_traffic = {
    'duration': np.random.exponential(scale=50, size=n_connections - n_attacks),
    'src_bytes': np.random.lognormal(mean=8, sigma=2, size=n_connections - n_attacks),
    'dst_bytes': np.random.lognormal(mean=7, sigma=2, size=n_connections - n_attacks),
    'num_failed_logins': np.random.poisson(lam=0.1, size=n_connections - n_attacks),
    'num_compromised': np.zeros(n_connections - n_attacks),
    'num_root': np.random.poisson(lam=0.05, size=n_connections - n_attacks),
    'num_file_creations': np.random.poisson(lam=2, size=n_connections - n_attacks),
    'count': np.random.poisson(lam=10, size=n_connections - n_attacks),
    'srv_count': np.random.poisson(lam=8, size=n_connections - n_attacks),
}

# Attack traffic (anomalous patterns)
attack_traffic = {
    'duration': np.random.choice([0, 0, 0, np.random.uniform(1000, 5000)], size=n_attacks),
    'src_bytes': np.random.choice([0, np.random.uniform(100000, 1000000)], size=n_attacks),
    'dst_bytes': np.random.choice([0, np.random.uniform(100000, 1000000)], size=n_attacks),
    'num_failed_logins': np.random.poisson(lam=5, size=n_attacks),
    'num_compromised': np.random.poisson(lam=3, size=n_attacks),
    'num_root': np.random.poisson(lam=2, size=n_attacks),
    'num_file_creations': np.random.poisson(lam=20, size=n_attacks),
    'count': np.random.poisson(lam=100, size=n_attacks),
    'srv_count': np.random.poisson(lam=5, size=n_attacks),
}

# Combine
df_network = pd.DataFrame({
    key: np.concatenate([normal_traffic[key], attack_traffic[key]])
    for key in normal_traffic.keys()
})

# Add categorical features
df_network['protocol_type'] = np.random.choice(['tcp', 'udp', 'icmp'], size=n_connections)
df_network['service'] = np.random.choice(['http', 'ftp', 'smtp', 'ssh', 'other'], size=n_connections)
df_network['flag'] = np.random.choice(['SF', 'S0', 'REJ', 'RSTR'], size=n_connections)

y_true_network = np.array([0] * (n_connections - n_attacks) + [1] * n_attacks)

print(f"Dataset shape: {df_network.shape}")
print(f"Attack rate: {100 * n_attacks / n_connections:.2f}%")
print(f"\nSample connections:\n{df_network.head()}")

In [ ]:
# Train intrusion detection system
detector_network = PSOD(
    cat_columns=['protocol_type', 'service', 'flag'],
    min_cols_chosen=0.5,
    max_cols_chosen=1.0,
    stdevs_to_outlier=2.0,
    transform_algorithm='yeo-johnson',
    contamination=0.03,
    random_seed=42
)

print("Training intrusion detection system...")
scores_network = detector_network.fit_predict(df_network, return_class=False)
labels_network = detector_network.fit_predict(df_network, return_class=True)

metrics_network = evaluate_outlier_detection(y_true_network, labels_network, scores_network)

print("\n" + "=" * 50)
print("INTRUSION DETECTION PERFORMANCE")
print("=" * 50)
print(f"Precision:  {metrics_network['precision']:.3f}")
print(f"Recall:     {metrics_network['recall']:.3f}")
print(f"F1-Score:   {metrics_network['f1']:.3f}")
print(f"ROC-AUC:    {metrics_network['roc_auc']:.3f}")
print("=" * 50)

print(f"\nDetected attacks: {sum(labels_network)}")
print(f"Actual attacks: {sum(y_true_network)}")
print(f"False alarms: {sum((labels_network == 1) & (y_true_network == 0))}")

## Case Study 3: IoT Sensor Monitoring

### Problem Description

Monitor multiple IoT sensors in real-time to detect equipment failures and anomalies.

### Challenge:
- Multivariate time series data
- Sensor drift and noise
- Early failure detection
- Different failure modes

In [ ]:
# Generate IoT sensor data
np.random.seed(42)
n_readings = 2000
n_anomalies = 50

timestamps = pd.date_range(start='2024-01-01', periods=n_readings, freq='5min')

# Simulate multiple sensors with correlations
temperature = 25 + 5 * np.sin(np.linspace(0, 4 * np.pi, n_readings)) + np.random.randn(n_readings) * 0.5
pressure = 100 + 2 * np.sin(np.linspace(0, 4 * np.pi, n_readings)) + np.random.randn(n_readings) * 0.3
vibration = 0.5 + 0.2 * np.sin(np.linspace(0, 8 * np.pi, n_readings)) + np.random.randn(n_readings) * 0.05
humidity = 60 + 10 * np.cos(np.linspace(0, 4 * np.pi, n_readings)) + np.random.randn(n_readings) * 1.0
power_consumption = 50 + 10 * np.sin(np.linspace(0, 4 * np.pi, n_readings)) + np.random.randn(n_readings) * 2.0

# Add anomalies (equipment failures)
anomaly_indices = np.random.choice(n_readings, n_anomalies, replace=False)
y_true_iot = np.zeros(n_readings)

for idx in anomaly_indices:
    y_true_iot[idx] = 1
    # Simulate different failure modes
    failure_type = np.random.choice(['overheat', 'pressure_spike', 'vibration', 'power_surge'])
    
    if failure_type == 'overheat':
        temperature[idx] += np.random.uniform(10, 20)
    elif failure_type == 'pressure_spike':
        pressure[idx] += np.random.uniform(20, 40)
    elif failure_type == 'vibration':
        vibration[idx] += np.random.uniform(2, 5)
    elif failure_type == 'power_surge':
        power_consumption[idx] += np.random.uniform(50, 100)

df_iot = pd.DataFrame({
    'timestamp': timestamps,
    'temperature': temperature,
    'pressure': pressure,
    'vibration': vibration,
    'humidity': humidity,
    'power': power_consumption,
})

print(f"IoT dataset shape: {df_iot.shape}")
print(f"Anomaly rate: {100 * n_anomalies / n_readings:.2f}%")
print(f"\nSensor readings:\n{df_iot.head()}")

In [ ]:
# Feature engineering for IoT data
df_iot_features = df_iot.copy()

# Rolling statistics
for col in ['temperature', 'pressure', 'vibration', 'humidity', 'power']:
    df_iot_features[f'{col}_rolling_mean'] = df_iot_features[col].rolling(window=10).mean()
    df_iot_features[f'{col}_rolling_std'] = df_iot_features[col].rolling(window=10).std()
    df_iot_features[f'{col}_diff'] = df_iot_features[col].diff()

# Fill NaN
df_iot_features = df_iot_features.fillna(method='bfill')

# Select features (exclude timestamp)
feature_cols = [col for col in df_iot_features.columns if col != 'timestamp']
X_iot = df_iot_features[feature_cols]

print(f"Total features: {X_iot.shape[1]}")

In [ ]:
# Train IoT anomaly detector
detector_iot = PSOD(
    min_cols_chosen=0.5,
    max_cols_chosen=1.0,
    stdevs_to_outlier=2.5,
    transform_algorithm='yeo-johnson',
    random_seed=42
)

print("Training IoT anomaly detector...")
scores_iot = detector_iot.fit_predict(X_iot, return_class=False)
labels_iot = detector_iot.fit_predict(X_iot, return_class=True)

metrics_iot = evaluate_outlier_detection(y_true_iot, labels_iot, scores_iot)

print("\n" + "=" * 50)
print("IOT ANOMALY DETECTION PERFORMANCE")
print("=" * 50)
print(f"Precision:  {metrics_iot['precision']:.3f}")
print(f"Recall:     {metrics_iot['recall']:.3f}")
print(f"F1-Score:   {metrics_iot['f1']:.3f}")
print(f"ROC-AUC:    {metrics_iot['roc_auc']:.3f}")
print("=" * 50)

In [ ]:
# Visualize IoT sensor monitoring
from psod.visualization import plot_timeseries_outliers

fig, axes = plt.subplots(5, 1, figsize=(16, 15))

sensors = ['temperature', 'pressure', 'vibration', 'humidity', 'power']
outlier_mask = labels_iot == 1

for idx, sensor in enumerate(sensors):
    axes[idx].plot(df_iot['timestamp'], df_iot[sensor], linewidth=1, alpha=0.7, label=sensor)
    axes[idx].scatter(
        df_iot.loc[outlier_mask, 'timestamp'],
        df_iot.loc[outlier_mask, sensor],
        c='red',
        s=100,
        marker='X',
        label='Detected Anomaly',
        zorder=5
    )
    axes[idx].set_ylabel(sensor.title())
    axes[idx].legend(loc='upper right')
    axes[idx].grid(True, alpha=0.3)
    
axes[0].set_title('IoT Sensor Monitoring with Anomaly Detection', fontsize=14, fontweight='bold')
axes[4].set_xlabel('Time')

plt.tight_layout()
plt.show()

## Summary and Deployment Considerations

### Key Learnings from Case Studies:

1. **Feature Engineering is Critical**
   - Domain knowledge improves detection
   - Derived features often more informative than raw features
   
2. **Context Matters**
   - Fraud detection: Minimize false positives (customer experience)
   - Intrusion detection: Minimize false negatives (security)
   - IoT monitoring: Early detection (prevent failures)

3. **Model Tuning**
   - Adjust contamination rate based on domain
   - Tune stdevs_to_outlier for sensitivity
   - Use appropriate transformations

### Deployment Checklist:

- ✅ Save trained models with `save_model()`
- ✅ Document feature engineering pipeline
- ✅ Set up monitoring for model drift
- ✅ Establish alerting thresholds
- ✅ Create human review process for high-risk detections
- ✅ Log predictions for auditing
- ✅ Plan for model retraining schedule
- ✅ Test edge cases and failure modes

### Production Code Example:

In [ ]:
# Clean up
import os

for file in ['fraud_detector.pkl']:
    if os.path.exists(file):
        os.remove(file)

print("Cleanup complete!")